# 114 — Ciclo ReAct y observación del entorno

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**ReAct** (Yao et al., arXiv:2210.03629) estructura cada iteración del agente en tres
elementos: **thought** (razonamiento en texto, sin efectos), **action** (invocación de
una herramienta con argumentos, o `finish`) y **observation** (la respuesta REAL del
entorno — el modelo no la genera: la recibe).

```text
repetir hasta finish o presupuesto agotado:
    thought_t     = LLM(contexto)             # razonar sobre lo observado
    action_t      = LLM(contexto + thought)   # decidir tool + args
    observation_t = entorno.ejecutar(action)  # información nueva y verificada
    contexto     += thought + action + observation
```

La observación es la única entrada de información nueva al bucle: sin ella el modelo
solo puede alucinar el estado del mundo (modo chain-of-thought). Los errores también
son observaciones — y de las más valiosas para el siguiente thought.

### 🔄 Lo que garantiza el patrón (bien implementado)

1. **Grounding:** cada decisión se toma sobre el último estado observado.
2. **Traza auditable:** la secuencia `(thought, action, observation)*` explica el porqué
   de cada paso — depurar un agente es leer su traza.
3. **Parada:** por decisión (`finish` con condiciones verificadas) o por presupuesto de
   pasos; nunca por "ya ejecuté lo que tenía pensado".

El laboratorio `agent` emite una traza de dos acciones (`status()` → obs
`{"healthy": true}`, `sum(7,5)` → obs `12`) con los thoughts implícitos: la condición
de éxito (`healthy == true` y `sum == 12`) se evalúa contra observaciones.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("agent", seed=114)
show(result)


## Reflexión

1. En la traza del laboratorio, ¿qué campo de cada paso proviene del entorno y cuál
   proviene de la política de decisión? ¿Por qué el grounding desaparecería si el mismo
   componente pudiera escribir ambos?
2. ReAct reduce la alucinación frente a chain-of-thought puro, pero el paper reporta
   errores de *reasoning* aun con observaciones correctas. ¿Qué aspecto de un thought
   temprano puede sesgar toda la trayectoria y qué mecanismo de la clase 115 lo mitiga?
3. Si `sum(7, 5)` devolviera `{"error": "timeout"}`, ¿qué debería contener el siguiente
   thought y por qué silenciar ese error en la herramienta "ciega" al agente?